# GS Map Full Run - Colab 1 GPU

Notebook nay danh cho Google Colab 1 GPU. Truoc khi chay: `Runtime` -> `Change runtime type` -> chon `GPU`.

Mac dinh chay dataset benchmark nho `blender-lego` tu NeRF Synthetic. Output se luu vao Google Drive: `/content/drive/MyDrive/gs-map-runs`.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/duy12i1i7/gs-map.git"
DATASET = "blender-lego"
ITERATIONS = "7000"
NUM_DEVICES = "1"
RUN_SMOKE_TEST = False
SMOKE_ITERATIONS = "1000"

# Chi dung external path khi doi DATASET ve "mill19-building".
# Neu ban da upload dataset san, dien path vao day.
# Vi du: "/content/drive/MyDrive/datasets/mill19/building"
EXTERNAL_DATASET_PATH = ""

from google.colab import drive
drive.mount("/content/drive")

WORK_ROOT = Path("/content")
RUN_ROOT = Path("/content/drive/MyDrive/gs-map-runs")
PROJECT_DIR = WORK_ROOT / "gs-map"
DATA_DIR = RUN_ROOT / "data"
OUTPUT_DIR = RUN_ROOT / "outputs"
EXPORT_DIR = RUN_ROOT / "exports"

for path in [RUN_ROOT, DATA_DIR, OUTPUT_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "PROJECT_DIR": str(PROJECT_DIR),
    "DATASET": DATASET,
    "ITERATIONS": ITERATIONS,
    "NUM_DEVICES": NUM_DEVICES,
    "RUN_SMOKE_TEST": "1" if RUN_SMOKE_TEST else "0",
    "SMOKE_ITERATIONS": SMOKE_ITERATIONS,
    "EXTERNAL_DATASET_PATH": EXTERNAL_DATASET_PATH,
    "DATA_DIR": str(DATA_DIR),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "EXPORT_DIR": str(EXPORT_DIR),
    "GSMAP_BACKEND": "native",
})

print("Project:", PROJECT_DIR)
print("Run root:", RUN_ROOT)
print("Dataset:", DATASET)
print("Iterations:", ITERATIONS)
print("Num devices:", NUM_DEVICES)

## Clone repo va cai dependencies

In [ ]:
if (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(["git", "status", "-sb"], check=True)

In [ ]:
%%bash
set -euo pipefail
cd "${PROJECT_DIR}"
python -m pip install -U pip
python -m pip install -r requirements.txt
ns-train splatfacto --help >/dev/null
./run.sh doctor

## Kiem tra GPU sau khi cai dependencies

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=False)
print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## Optional smoke test

In [ ]:
%%bash
set -euo pipefail
cd "${PROJECT_DIR}"
if [ "${RUN_SMOKE_TEST}" = "1" ]; then
  ./run.sh download synthetic-map
  env ITERATIONS="${SMOKE_ITERATIONS}" NUM_DEVICES=1 ./run.sh train synthetic-map
else
  echo "Skipping smoke test."
fi

## Chuan bi dataset

In [ ]:
external = os.environ.get("EXTERNAL_DATASET_PATH", "").strip()
if external:
    if DATASET != "mill19-building":
        raise ValueError("EXTERNAL_DATASET_PATH chi ap dung cho mill19-building. Clear no hoac doi DATASET.")
    external_path = Path(external)
    if not external_path.exists():
        raise FileNotFoundError(external_path)
    target = Path(os.environ["DATA_DIR"]) / "mill19" / "building"
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        print(f"Using existing dataset target: {target}")
    else:
        target.symlink_to(external_path, target_is_directory=True)
        print(f"Linked {target} -> {external_path}")
else:
    subprocess.run(["bash", "-lc", f"cd '{PROJECT_DIR}' && ./run.sh download '{DATASET}'"], check=True)

## Train full voi 1 GPU

In [ ]:
%%bash
set -euo pipefail
cd "${PROJECT_DIR}"
env ITERATIONS="${ITERATIONS}" NUM_DEVICES=1 ./run.sh train "${DATASET}"

## Export va eval

In [ ]:
%%bash
set -euo pipefail
cd "${PROJECT_DIR}"
./run.sh export "${DATASET}"
./run.sh eval "${DATASET}" || true
find "${OUTPUT_DIR}" -path '*/config.yml' -type f | sort
find "${EXPORT_DIR}" -type f | sort